In [1]:
# Import necessary dependencies
import warnings
from pathlib import Path
from typing import Dict

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score,
    confusion_matrix
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier

from dotenv import load_dotenv
load_dotenv()

from extract_features import bert_encode_sentences, openai_encode_sentences

# Set seed for reproducibility
RANDOM_SEED     = 1567640
DATA_PATH = "./data.csv"

warnings.filterwarnings("ignore")

In [2]:
# Build the text feature pipeline based on the method name
def build_pipeline(method: str, random_state: int):
    if method == "tfidf":
        return Pipeline([
            ("tfidf", TfidfVectorizer(max_features=200, stop_words="english")),
            # keep sparse matrix
            # ("dense", FunctionTransformer(lambda X: X.toarray(), accept_sparse=True)),
            # ("pca",   PCA(n_components=200, random_state=random_state))
        ])
    elif method == "bert":
        return Pipeline([
            ("bert",   FunctionTransformer(bert_encode_sentences, validate=False)),
            ("pca",    PCA(n_components=200, random_state=random_state))
        ])
    elif method == "openai":
        return Pipeline([
            ("openai", FunctionTransformer(openai_encode_sentences, validate=False)),
            ("pca",    PCA(n_components=200, random_state=random_state))
        ])
    else:
        raise ValueError(f"Unknown method {method}")

In [3]:
# Model & Grid Search
model_params = {
    "Random":       [{}],
    "Zero R":       [{}],
    "Random Forest":[{"max_depth": [d]} for d in range(1, 50, 2)],
    "KNN":          [{"n_neighbors": [n]} for n in range(1, 50, 2)],
    "SGDClassifier":[{"alpha": [10**i]} for i in range(-6, 0)]
}

def init_model(name: str, params: Dict):
    if name == "Random":
        return DummyClassifier(strategy="uniform", random_state=RANDOM_SEED)
    if name == "Zero R":
        return DummyClassifier(strategy="most_frequent", random_state=RANDOM_SEED)
    if name == "Random Forest":
        return RandomForestClassifier(random_state=RANDOM_SEED, class_weight='balanced', criterion='entropy', n_estimators=100, n_jobs=1, **params)
    if name == "KNN":
        return KNeighborsClassifier(weights='distance', metric='cosine', **params)
    if name == "SGDClassifier":
        return SGDClassifier(loss="log_loss", class_weight='balanced', max_iter=10000, random_state=RANDOM_SEED, **params)
    raise ValueError(name)

In [4]:
# calculate metrics: accuracy/precision/recall/f1/auc
def calculate_metrics(y_true, y_pred, y_proba=None) -> Dict:
    res = {
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall":    recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1":        f1_score(y_true, y_pred, average="weighted", zero_division=0)
    }
    if y_proba is not None:
        try:
            if y_proba.ndim == 2 and y_proba.shape[1] == 2:
                res["auc"] = roc_auc_score(y_true, y_proba[:, 1])
            else:
                res["auc"] = roc_auc_score(y_true, y_proba,
                                           multi_class="ovr", average="weighted")
        except ValueError:
            res["auc"] = np.nan
    else:
        res["auc"] = np.nan
    return res

In [5]:
# train and evaluate with nest cross validation
def train_and_evaluate(X, y, text_transform, cat_cols, num_cols):
    outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
    results = []

    for outer_fold, (train_id, test_id) in enumerate(outer.split(X, y), 1):
        X_train, X_test = X.iloc[train_id], X.iloc[test_id]
        y_train, y_test = y.iloc[train_id], y.iloc[test_id]

        preprocess = ColumnTransformer(transformers=[
            ("text", text_transform, "Text"),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
            ("num", Pipeline([("sc", StandardScaler())]), num_cols)
        ], remainder="drop")

        for model_name, grid in model_params.items():
            if len(grid) > 1:
                estimator = GridSearchCV(
                    estimator=init_model(model_name, grid[0]),
                    param_grid=grid,
                    cv=inner,
                    scoring="f1_weighted",
                    n_jobs=1
                )
            else:
                estimator = init_model(model_name, grid[0])

            pipe = Pipeline(steps=[("preprocess", preprocess), ("clf", estimator)])
            pipe.fit(X_train, y_train)

            if isinstance(estimator, GridSearchCV):
                best_est = pipe.named_steps["clf"].best_estimator_
                best_params = pipe.named_steps["clf"].best_params_
            else:
                best_est = pipe.named_steps["clf"]
                best_params = {}

            y_pred = pipe.predict(X_test)
            if hasattr(best_est, "predict_proba"):
                y_proba = best_est.predict_proba(pipe.named_steps["preprocess"].transform(X_test))
            else:
                y_proba = None

            metrics = calculate_metrics(y_test, y_pred, y_proba)
            conf_mat = confusion_matrix(y_test, y_pred)
            metrics.update({
                "outer_fold":  outer_fold,
                "model":       model_name,
                "best_params": best_params,
                "confusion_matrix": conf_mat
            })
            results.append(metrics)
            print(
                f"[Outer Fold {outer_fold}] {model_name} – "
                f"Accuracy={metrics['accuracy']:.4f}, Precision={metrics['precision']:.4f}, "
                f"F1={metrics['f1']:.4f}, Recall={metrics['recall']:.4f}, AUC={metrics['auc']:.4f}, "
                f"Best Params: {best_params}, Confusion Matrix: {conf_mat}, "
            )
    return pd.DataFrame(results)

In [6]:
# Load data and preprocessess, show the first 10 rows
if not Path(DATA_PATH).exists():
    raise FileNotFoundError(f"{DATA_PATH} not found.")
df = pd.read_csv(DATA_PATH).dropna(axis=0, how='any')
df["Text"] = (df["Title"].fillna("") + " " + df["Review Text"].fillna("")).str.strip()
df.drop(columns=["Title", "Review Text", "Rating"], inplace=True)
display(df.head(10))
# Show first 10 rows of data

,Unnamed: 0,Clothing ID,Age,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name,Text
2,2,1077,60,0,0,General,Dresses,Dresses,Some major design flaws I had such high hopes ...
3,3,1049,50,1,0,General Petite,Bottoms,Pants,"My favorite buy! I love, love, love this jumps..."
4,4,847,47,1,6,General,Tops,Blouses,Flattering shirt This shirt is very flattering...
5,5,1080,49,0,4,General,Dresses,Dresses,Not for the very petite I love tracy reese dre...
6,6,858,39,1,1,General Petite,Tops,Knits,Cagrcoal shimmer fun I aded this in my basket ...
7,7,858,39,1,4,General Petite,Tops,Knits,"Shimmer, surprisingly goes with lots I ordered..."
8,8,1077,24,1,0,General,Dresses,Dresses,Flattering I love this dress. i usually get an...
9,9,1077,34,1,0,General,Dresses,Dresses,"Such a fun dress! I'm 5""5' and 125 lbs. i orde..."
10,10,1077,53,0,14,General,Dresses,Dresses,Dress looks like it's made of cheap material D...
12,12,1095,53,1,2,General Petite,Dresses,Dresses,Perfect!!! More and more i find myself reliant...


In [7]:
cat_cols = ["Division Name", "Department Name", "Class Name"]
num_cols = ["Age"]
text_methods = {
    "openai": build_pipeline("openai", random_state=RANDOM_SEED),
    "bert": build_pipeline("bert", random_state=RANDOM_SEED),
    "tfidf": build_pipeline("tfidf", random_state=RANDOM_SEED),
}

results = []
for name, text_transform in text_methods.items():
    print(f"================ Evaluating feature: {name.upper()} ==================")
    res = train_and_evaluate(df, df["Recommended IND"], text_transform, cat_cols, num_cols)
    res["feature"] = name
    results.append(res)
    print("\n\n")

final = pd.concat(results, ignore_index=True)
final.to_csv("./final.csv", index=False)
# print intermediate results of training

================ Evaluating feature: OPENAI ==================
[Outer Fold 1] Random – Accuracy=0.4963, Precision=0.6924, F1=0.5534, Recall=0.4963, AUC=0.5000, Best Params: {}, Confusion Matrix: [[ 330  385]
 [1596 1622]], 
[Outer Fold 1] Zero R – Accuracy=0.8182, Precision=0.6695, F1=0.7364, Recall=0.8182, AUC=0.5000, Best Params: {}, Confusion Matrix: [[   0  715]
 [   0 3218]], 
[Outer Fold 1] Random Forest – Accuracy=0.9156, Precision=0.9219, F1=0.9178, Recall=0.9156, AUC=0.9623, Best Params: {'max_depth': 9}, Confusion Matrix: [[ 603  112]
 [ 220 2998]], 
[Outer Fold 1] KNN – Accuracy=0.8980, Precision=0.8927, F1=0.8910, Recall=0.8980, AUC=0.9426, Best Params: {'n_neighbors': 9}, Confusion Matrix: [[ 411  304]
 [  97 3121]], 
[Outer Fold 1] SGDClassifier – Accuracy=0.9169, Precision=0.9351, F1=0.9214, Recall=0.9169, AUC=0.9773, Best Params: {'alpha': 0.0001}, Confusion Matrix: [[ 677   38]
 [ 289 2929]], 
[Outer Fold 2] Random – Accuracy=0.5095, Precision=0.7055, F1=0.5651, Recall

In [8]:
final_agg = final.groupby(["feature", "model"]).agg(
    accuracy_mean   = ("accuracy",  "mean"),
    accuracy_std    = ("accuracy",  "std"),
    precision_mean  = ("precision", "mean"),
    precision_std   = ("precision", "std"),
    recall_mean     = ("recall",    "mean"),
    recall_std      = ("recall",    "std"),
    f1_mean         = ("f1",        "mean"),
    f1_std          = ("f1",        "std"),
    auc_mean        = ("auc",       "mean"),
    auc_std         = ("auc",       "std"),
).round(4)
display(final_agg)
# show train and test metrics table

accuracy_mean  accuracy_std  precision_mean  \
feature model                                                        
bert    KNN                   0.8576        0.0036          0.8442   
        Random                0.5068        0.0067          0.7028   
        Random Forest         0.8453        0.0051          0.8603   
        SGDClassifier         0.8663        0.0084          0.9000   
        Zero R                0.8182        0.0000          0.6694   
openai  KNN                   0.9013        0.0041          0.8965   
        Random                0.5068        0.0067          0.7028   
        Random Forest         0.9167        0.0029          0.9219   
        SGDClassifier         0.9190        0.0043          0.9338   
        Zero R                0.8182        0.0000          0.6694   
tfidf   KNN                   0.8111        0.0047          0.7794   
        Random                0.5068        0.0067          0.7028   
        Random Forest         0.8575        0.0013          0.8505   
        SGDClassifier         0.8381        0.0084          0.8765   
        Zero R                0.8182        0.0000          0.6694   

                       precision_std  recall_mean  recall_std  f1_mean  \
feature model                                                            
bert    KNN                   0.0055       0.8576      0.0036   0.8422   
        Random                0.0066       0.5068      0.0067   0.5627   
        Random Forest         0.0030       0.8453      0.0051   0.8512   
        SGDClassifier         0.0029       0.8663      0.0084   0.8755   
        Zero R                0.0000       0.8182      0.0000   0.7364   
openai  KNN                   0.0045       0.9013      0.0041   0.8966   
        Random                0.0066       0.5068      0.0067   0.5627   
        Random Forest         0.0039       0.9167      0.0029   0.9186   
        SGDClassifier         0.0033       0.9190      0.0043   0.9229   
        Zero R                0.0000       0.8182      0.0000   0.7364   
tfidf   KNN                   0.0053       0.8111      0.0047   0.7875   
        Random                0.0066       0.5068      0.0067   0.5627   
        Random Forest         0.0046       0.8575      0.0013   0.8528   
        SGDClassifier         0.0044       0.8381      0.0084   0.8496   
        Zero R                0.0000       0.8182      0.0000   0.7364   

                       f1_std  auc_mean  auc_std  
feature model                                     
bert    KNN            0.0027    0.8535   0.0172  
        Random         0.0059    0.5000   0.0000  
        Random Forest  0.0042    0.8900   0.0061  
        SGDClassifier  0.0069    0.9373   0.0024  
        Zero R         0.0000    0.5000   0.0000  
openai  KNN            0.0048    0.9308   0.0097  
        Random         0.0059    0.5000   0.0000  
        Random Forest  0.0028    0.9650   0.0018  
        SGDClassifier  0.0037    0.9745   0.0035  
        Zero R         0.0000    0.5000   0.0000  
tfidf   KNN            0.0050    0.6821   0.0098  
        Random         0.0059    0.5000   0.0000  
        Random Forest  0.0038    0.8831   0.0036  
        SGDClassifier  0.0065    0.9051   0.0040  
        Zero R         0.0000    0.5000   0.0000